# LifeMate baseline v2 on Kaggle

Runs PhoWhisper-medium ASR only, Qwen3-4B workflow with gold train text, and the complete ASR + workflow baseline. The run writes WER/CER, state-hash evaluation, tool-call traces, context-token lengths, iteration counts and error cases.

Enable GPU and Internet, then attach the extracted LifeMate data and the baseline_v2 code as Kaggle inputs. The code discovers train data automatically. Set LIMIT=0 for all train samples, or set SAMPLE_ID to run one train case with the gold transcript. The default vLLM profile targets memory-limited Kaggle GPUs; vLLM is not guaranteed to use less memory on every GPU.


### Batch inference
Để lấy số liệu nhanh: chọn `MODE="workflow"`, `INFERENCE_WORKERS=4`, `VLLM_MAX_NUM_SEQS=4`. Các mẫu dùng ToolEnv riêng; vLLM gom các node sẵn sàng. Giảm workers xuống 2 nếu thiếu VRAM.
Batch tăng throughput, không bảo đảm giảm latency từng mẫu. Đo giới hạn 30 giây bằng workers=1 trên A30. Batch có thể đổi output dù seed giống nhau; so sánh cùng cấu hình. Summary có `experiment_wall_s` và `samples_per_second`.


In [ ]:
from pathlib import Path

MODE = "all"   # "asr", "workflow", "baseline", or "all"
LIMIT = 50     # 0 = full train split
SAMPLE_ID = None  # Example: "train_1703"; runs only this case in workflow mode
SEED = 42
DATA_ROOT = None  # Example: "/kaggle/input/lifemate/train+public_test"
OUTPUT_DIR = "/kaggle/working/lifemate_baseline_v2"
RUN_MODE = "workflow" if SAMPLE_ID else MODE
RUN_OUTPUT_DIR = str(Path(OUTPUT_DIR) / "single" / SAMPLE_ID) if SAMPLE_ID else OUTPUT_DIR
ASR_MODEL = "vinai/PhoWhisper-medium"
LLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LLM_BACKEND = "vllm"  # Change to "transformers" for the old backend
LLM_TEMPERATURE = 0.7
ASR_BATCH_SIZE = 1
# 1: isolated latency; 4: batch ready node requests from four independent samples.
INFERENCE_WORKERS = 1  # Try 4 to collect results faster; reduce to 2 if VRAM is tight
VLLM_MAX_NUM_SEQS = INFERENCE_WORKERS
BATCH_WAIT_MS = 10
STRUCTURED_OUTPUTS = True  # guidance backend; schema compilation preflight runs before model load
VLLM_USE_CUDA_GRAPHS = False
VLLM_MAX_MODEL_LEN = 12288
VLLM_GPU_MEMORY_UTILIZATION = 0.82
VLLM_TENSOR_PARALLEL_SIZE = 1  # Set to 2 only when two GPUs are visible
VLLM_ATTENTION_BACKEND = "TRITON_ATTN"  # Avoid FlashInfer JIT linker errors on Kaggle T4
VLLM_DTYPE = "float16"  # Tesla T4 does not support bfloat16
MAX_PROMPT_TOKENS = 11500
MAX_NEW_TOKENS = 512
MAX_NO_PROGRESS = 2

paths = list(Path("/kaggle/input").rglob("baseline_v2/run.py")) if Path("/kaggle/input").exists() else []
paths += list(Path.cwd().rglob("baseline_v2/run.py"))
if not paths:
    raise FileNotFoundError("Attach the baseline_v2 folder or set SCRIPT_DIR manually.")
SCRIPT_DIR = paths[0].parent
required_files = ("agent.py", "agent_pipeline.py", "pipeline_rules.py", "evidence_rules.py", "intent_contract.py", "semantic_checks.py", "output_contracts.py", "inference_errors.py", "batch_inference.py", "node_reviews.py", "source_contract.py")
if SAMPLE_ID:
    required_files += ("run_one.py",)
for required_file in required_files:
    if not (SCRIPT_DIR / required_file).is_file():
        raise FileNotFoundError(f"The attached baseline_v2 is outdated: {required_file} is missing. Re-upload the updated folder.")
RUN_SCRIPT = SCRIPT_DIR / ("run_one.py" if SAMPLE_ID else "run.py")
print("Script:", RUN_SCRIPT)
print("Output:", RUN_OUTPUT_DIR)


In [ ]:
import subprocess
import sys

requirements = "requirements-vllm.txt" if LLM_BACKEND == "vllm" and RUN_MODE != "asr" else "requirements.txt"
requirement_path = Path(SCRIPT_DIR) / requirements
if not requirement_path.is_file():
    raise FileNotFoundError(requirement_path)
install_command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "on", "-r", str(requirement_path)]
print("Installing:", requirement_path, flush=True)
subprocess.run(install_command, check=True, timeout=1800)
print("Dependency installation completed.")


In [ ]:
import os
import subprocess
import sys

command = [sys.executable, str(RUN_SCRIPT)]
if SAMPLE_ID:
    command.append(SAMPLE_ID)
else:
    command.extend(["--mode", MODE, "--limit", str(LIMIT)])
command.extend([
    "--seed", str(SEED),
    "--output-dir", RUN_OUTPUT_DIR,
    "--asr-model", ASR_MODEL,
    "--llm-model", LLM_MODEL,
    "--llm-backend", LLM_BACKEND,
    "--llm-temperature", str(LLM_TEMPERATURE),
    "--asr-batch-size", str(ASR_BATCH_SIZE),
    "--inference-workers", str(1 if SAMPLE_ID else INFERENCE_WORKERS),
    "--vllm-max-num-seqs", str(VLLM_MAX_NUM_SEQS),
    "--batch-wait-ms", str(BATCH_WAIT_MS),
    "--vllm-max-model-len", str(VLLM_MAX_MODEL_LEN),
    "--vllm-gpu-memory-utilization", str(VLLM_GPU_MEMORY_UTILIZATION),
    "--vllm-tensor-parallel-size", str(VLLM_TENSOR_PARALLEL_SIZE),
    "--vllm-attention-backend", VLLM_ATTENTION_BACKEND,
    "--vllm-dtype", VLLM_DTYPE,
    "--max-prompt-tokens", str(MAX_PROMPT_TOKENS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-no-progress", str(MAX_NO_PROGRESS),
])
if not STRUCTURED_OUTPUTS:
    command.append("--no-structured-outputs")
if VLLM_USE_CUDA_GRAPHS:
    command.append("--vllm-use-cuda-graphs")
if DATA_ROOT:
    command.extend(["--data-root", DATA_ROOT])
run_env = os.environ.copy()
run_env["PYTHONHASHSEED"] = str(SEED)
run_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
if LLM_BACKEND == "vllm" and RUN_MODE != "asr":
    run_env["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
print("Running:", " ".join(command))
subprocess.run(command, check=True, env=run_env)


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

out = Path(RUN_OUTPUT_DIR)
for name in ("asr", "workflow_gold", "baseline_asr"):
    summary = out / name / "summary.json"
    chart = out / name / "charts.png"
    if summary.exists():
        print("\n", name)
        print(json.dumps(json.loads(summary.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    if chart.exists():
        display(Image(filename=str(chart)))
print("Download result folder:", out)
